# Phase 0 — Data Orientation & Diagnostic

**Dataset:** Student Wellness & Academic Performance  
**File:** `dataset/student_wellness.csv`  
**Date:** 2026-04-13  

---

This notebook reproduces the full Phase 0 diagnostic: structural overview of every column, data quality issue inventory, and diagnostic visualizations. By the end, we will know exactly what is broken and what cleaning decisions to make.

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import Markdown, display

# ── Pastel palette (from agent.md) ──
PALETTE = {
    "primary":    "#A8C8E8",
    "secondary":  "#F4A8B0",
    "accent1":    "#A8E8C8",
    "accent2":    "#F4D8A8",
    "accent3":    "#C8A8E8",
    "accent4":    "#F4F4A8",
    "neutral":    "#D0D0D0",
    "dark_text":  "#4A4A4A",
    "background": "#FAFAFA",
}
CATEGORICAL_COLORS = [
    "#A8C8E8", "#F4A8B0", "#A8E8C8", "#F4D8A8",
    "#C8A8E8", "#F4F4A8", "#B8D8F8", "#F8C8D8",
]

LAYOUT_DEFAULTS = dict(
    template="plotly_white",
    font=dict(family="Inter, Arial, sans-serif", size=13, color="#4A4A4A"),
    plot_bgcolor="#FAFAFA",
    paper_bgcolor="#FFFFFF",
    margin=dict(t=80, b=60, l=60, r=40),
)

def styled(fig, title):
    fig.update_layout(
        **LAYOUT_DEFAULTS,
        title=dict(text=title, font=dict(size=16, color="#4A4A4A"), x=0.5, xanchor="center"),
    )
    return fig

print("Libraries loaded ✓")

Libraries loaded ✓


## 1. Load the Raw Dataset

In [3]:
df = pd.read_csv("../dataset/student_wellness.csv")
print(f"Dataset: {df.shape[0]} rows × {df.shape[1]} columns")
df.head()

Dataset: 535 rows × 21 columns


,student_id,age,gender,major,year_in_school,gpa,study_hours_per_day,attendance_rate,sleep_hours_per_night,exercise_days_per_week,...,social_media_hours,caffeine_mg_per_day,stress_level,anxiety_score,depression_score,life_satisfaction,num_clubs,on_campus,has_part_time_job,monthly_spending
0,STU0118,19.0,Male,Psychology,2.0,2.53,3.3,96.1,9.1,4.0,...,2.8,0.0,3.4,3.0,0.0,7.1,1.0,False,No,998.98
1,STU0133,18.0,Female,Mechanical Engineering,1.0,3.17,6.1,87.1,4.5,1.0,...,2.2,90.0,7.8,9.0,9.0,2.5,2.0,True,Yes,666.49
2,STU0155,21.0,Male,Nursing,2.0,3.47,8.8,81.6,6.5,3.0,...,2.2,316.0,3.4,2.0,NaN,9.2,1.0,YES,Yes,1112.27
3,STU0246,18.0,Female,Business,2.0,3.11,7.6,96.1,7.6,NaN,...,1.7,236.0,4.6,5.0,4.0,7.4,3.0,True,Yes,958.16
4,STU0085,23.0,Female,Computer Science,4.0,2.40,5.5,91.7,8.3,0.0,...,2.9,124.0,4.8,5.0,5.0,5.5,3.0,True,No,969.36


In [22]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 535 entries, 0 to 534
Data columns (total 21 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   student_id              535 non-null    str    
 1   age                     535 non-null    float64
 2   gender                  535 non-null    str    
 3   major                   535 non-null    str    
 4   year_in_school          535 non-null    float64
 5   gpa                     493 non-null    float64
 6   study_hours_per_day     535 non-null    float64
 7   attendance_rate         535 non-null    float64
 8   sleep_hours_per_night   482 non-null    float64
 9   exercise_days_per_week  503 non-null    float64
 10  screen_time_hours       535 non-null    float64
 11  social_media_hours      535 non-null    float64
 12  caffeine_mg_per_day     498 non-null    float64
 13  stress_level            535 non-null    str    
 14  anxiety_score           466 non-null    float64
 15  

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 535 entries, 0 to 534
Data columns (total 21 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   student_id              535 non-null    str    
 1   age                     535 non-null    float64
 2   gender                  535 non-null    str    
 3   major                   535 non-null    str    
 4   year_in_school          535 non-null    float64
 5   gpa                     493 non-null    float64
 6   study_hours_per_day     535 non-null    float64
 7   attendance_rate         535 non-null    float64
 8   sleep_hours_per_night   482 non-null    float64
 9   exercise_days_per_week  503 non-null    float64
 10  screen_time_hours       535 non-null    float64
 11  social_media_hours      535 non-null    float64
 12  caffeine_mg_per_day     498 non-null    float64
 13  stress_level            535 non-null    str    
 14  anxiety_score           466 non-null    float64
 15  

## 2. Structural Overview — All 21 Columns

For each column we compute:
- pandas dtype (before any conversion)
- Missing count and percentage
- Number of unique values
- For numeric: min, max, mean, median, std
- For string/object: unique values (if ≤ 20) or top 10 most frequent

In [5]:
overview_rows = []

for col in df.columns:
    s = df[col]
    info = {
        "column": col,
        "pandas_dtype": str(s.dtype),
        "n_missing": int(s.isna().sum()),
        "pct_missing": round(s.isna().mean() * 100, 2),
        "n_unique": int(s.nunique()),
    }

    if pd.api.types.is_numeric_dtype(s):
        info["min"]    = round(float(s.min()), 4) if s.notna().any() else None
        info["max"]    = round(float(s.max()), 4) if s.notna().any() else None
        info["mean"]   = round(float(s.mean()), 4) if s.notna().any() else None
        info["median"] = round(float(s.median()), 4) if s.notna().any() else None
        info["std"]    = round(float(s.std()), 4) if s.notna().any() else None
        info["unique_values"] = None
    else:
        vc = s.dropna().value_counts()
        if len(vc) <= 20:
            info["unique_values"] = vc.to_dict()
        else:
            info["unique_values"] = vc.head(10).to_dict()
        info["min"] = info["max"] = info["mean"] = info["median"] = info["std"] = None

    overview_rows.append(info)

overview_df = pd.DataFrame(overview_rows)
overview_df

,column,pandas_dtype,n_missing,pct_missing,n_unique,unique_values,min,max,mean,median,std
0,student_id,str,0,0.00,520,"{'STU0056': 2, 'STU0360': 2, 'STU0212': 2, 'ST...",NaN,NaN,NaN,NaN,NaN
1,age,float64,0,0.00,14,None,-3.0,999.00,23.7888,21.00,43.3858
2,gender,str,0,0.00,12,"{'Female': 206, 'Male': 201, 'Non-binary': 34,...",NaN,NaN,NaN,NaN,NaN
3,major,str,0,0.00,10,"{'Business': 81, 'Computer Science': 77, 'Psyc...",NaN,NaN,NaN,NaN,NaN
4,year_in_school,float64,0,0.00,4,None,1.0,4.00,2.4056,2.00,1.1142
5,gpa,float64,42,7.85,180,None,-1.0,6.00,3.0733,3.12,0.5799
6,study_hours_per_day,float64,0,0.00,91,None,-2.0,35.00,7.0241,7.00,2.6986
7,attendance_rate,float64,0,0.00,282,None,50.0,128.00,82.8505,82.80,12.1247
8,sleep_hours_per_night,float64,53,9.91,65,None,-3.0,24.00,6.8311,6.80,1.7141
9,exercise_days_per_week,float64,32,5.98,8,None,0.0,7.00,3.4453,3.00,2.2689


### Numeric columns summary

In [6]:
numeric_cols = overview_df[overview_df["min"].notna()][["column", "pandas_dtype", "n_missing", "pct_missing", "n_unique", "min", "max", "mean", "median", "std"]]
numeric_cols.set_index("column")

,pandas_dtype,n_missing,pct_missing,n_unique,min,max,mean,median,std
column,,,,,,,,,
age,float64,0,0.00,14,-3.0,999.00,23.7888,21.00,43.3858
year_in_school,float64,0,0.00,4,1.0,4.00,2.4056,2.00,1.1142
gpa,float64,42,7.85,180,-1.0,6.00,3.0733,3.12,0.5799
study_hours_per_day,float64,0,0.00,91,-2.0,35.00,7.0241,7.00,2.6986
attendance_rate,float64,0,0.00,282,50.0,128.00,82.8505,82.80,12.1247
sleep_hours_per_night,float64,53,9.91,65,-3.0,24.00,6.8311,6.80,1.7141
exercise_days_per_week,float64,32,5.98,8,0.0,7.00,3.4453,3.00,2.2689
screen_time_hours,float64,0,0.00,95,1.4,14.40,7.5624,7.60,2.0028
social_media_hours,float64,0,0.00,56,0.0,6.10,2.5546,2.50,1.1917


### String / object columns — unique values

In [7]:
string_cols = overview_df[overview_df["unique_values"].notna()]
for _, row in string_cols.iterrows():
    print(f"\n▸ {row['column']}  (dtype={row['pandas_dtype']}, missing={row['n_missing']} ({row['pct_missing']}%), unique={row['n_unique']})")
    for val, count in row["unique_values"].items():
        print(f"    {val!r:30s} → {count}")


▸ student_id  (dtype=str, missing=0 (0.0%), unique=520)
    'STU0056'                      → 2
    'STU0360'                      → 2
    'STU0212'                      → 2
    'STU0186'                      → 2
    'STU0126'                      → 2
    'STU0340'                      → 2
    'STU0425'                      → 2
    'STU0366'                      → 2
    'STU0061'                      → 2
    'STU0166'                      → 2

▸ gender  (dtype=str, missing=0 (0.0%), unique=12)
    'Female'                       → 206
    'Male'                         → 201
    'Non-binary'                   → 34
    'Prefer not to say'            → 19
    'M'                            → 12
    'F'                            → 12
    'man'                          → 11
    'FEMALE'                       → 10
    'male'                         → 10
    'female'                       → 7
    'MALE'                         → 7
    'woman'                        → 6

▸ major  (dtype=str, 

## 3. Duplicate Row Check

The dataset is expected to contain ~520 unique students but has 535 rows. Let's find exact duplicates.

In [8]:
dups = df.duplicated()
n_dups = int(dups.sum())
print(f"Exact duplicate rows: {n_dups}")
if n_dups:
    print(f"Indices: {list(df.index[dups])}")
    display(df[dups])

Exact duplicate rows: 3
Indices: [189, 404, 488]


,student_id,age,gender,major,year_in_school,gpa,study_hours_per_day,attendance_rate,sleep_hours_per_night,exercise_days_per_week,...,social_media_hours,caffeine_mg_per_day,stress_level,anxiety_score,depression_score,life_satisfaction,num_clubs,on_campus,has_part_time_job,monthly_spending
189,STU0061,25.0,Non-binary,Computer Science,3.0,3.14,3.7,55.1,7.5,7.0,...,4.5,214.0,3.2,3.0,4.0,10.0,2.0,True,No,423.91
404,STU0425,21.0,Female,Computer Science,2.0,2.69,4.5,68.2,4.7,5.0,...,2.7,281.0,8.2,12.0,6.0,3.7,4.0,True,Yes,1370.29
488,STU0234,24.0,Male,Communications,3.0,3.69,9.6,92.7,7.4,1.0,...,2.8,214.0,4.6,5.0,5.0,5.2,4.0,False,No,860.43


In [9]:
# Duplicate rows pie chart
if n_dups:
    fig = go.Figure(go.Pie(
        labels=["Unique Rows", "Duplicate Rows"],
        values=[len(df) - n_dups, n_dups],
        marker_colors=[PALETTE["primary"], PALETTE["secondary"]],
        textinfo="label+value+percent",
    ))
    styled(fig, f"Duplicate Rows: {n_dups} of {len(df)}")
    fig.show()

**So what?** 3 exact duplicate rows (0.6%) — modest inflation. These should be dropped (keep first occurrence) before any analysis.

## 4. Missing Values Overview

397 total missing cells across 9 columns (3.5% of all cells).

In [10]:
miss = df.isnull().sum()
miss = miss[miss > 0].sort_values(ascending=True)

fig = go.Figure(go.Bar(
    y=miss.index, x=miss.values, orientation="h",
    marker_color=PALETTE["secondary"],
    text=[f"{v} ({v/len(df)*100:.1f}%)" for v in miss.values],
    textposition="outside",
))
styled(fig, "Missing Values per Column")
fig.update_xaxes(title_text="Number of Missing Values")
fig.show()

| Tier | Columns | % Missing |
|------|---------|----------|
| High | anxiety_score, depression_score | 12.9% |
| Medium | sleep_hours_per_night (9.9%), monthly_spending (9.0%), gpa (7.9%), caffeine_mg_per_day (6.9%), exercise_days_per_week (6.0%) | 6–10% |
| Low | has_part_time_job (4.9%), num_clubs (3.9%) | < 5% |
| None | 12 columns | 0% |

**Key question for Phase 1:** Do anxiety_score and depression_score share the same missing rows? If so, this suggests MNAR (Missing Not At Random).

## 5. Column-by-Column Diagnostic Charts

One diagnostic chart per problematic column, with impossible / out-of-range values highlighted.

### 5.1 age — Impossible values

6 impossible ages found: −3, 0, 5, 150, 200, 999. The mean (23.8) is artificially inflated; the median (21) reflects the true center.

In [11]:
age_vals = pd.to_numeric(df["age"], errors="coerce")
impossible_ages = age_vals[(age_vals < 16) | (age_vals > 60)].dropna()

print(f"Impossible age values: {sorted(impossible_ages.unique().tolist())}")
print(f"Rows affected: {len(impossible_ages)}")
print(f"Mean (raw): {age_vals.mean():.2f}  |  Median: {age_vals.median():.1f}  |  Std: {age_vals.std():.2f}")

fig = go.Figure(go.Histogram(x=age_vals, nbinsx=50, marker_color=PALETTE["primary"]))
for a in impossible_ages.unique():
    fig.add_vline(x=a, line_dash="dash", line_color="#F4A8B0",
                  annotation_text=f"impossible: {a}")
styled(fig, "Age Distribution (impossible values highlighted)")
fig.update_xaxes(title_text="Age (years)")
fig.update_yaxes(title_text="Count")
fig.show()

Impossible age values: [-3.0, 0.0, 5.0, 150.0, 200.0, 999.0]
Rows affected: 6
Mean (raw): 23.79  |  Median: 21.0  |  Std: 43.39


**So what?** The huge std (43.4) is entirely driven by 6 bad rows. After setting these to NaN the valid range is 18–25, consistent with a college sample.

### 5.2 gender — Inconsistent encoding

75 rows (14%) use non-canonical encodings across 12 variants.

In [12]:
gender_vc = df["gender"].value_counts()
print("Gender value counts:")
for val, cnt in gender_vc.items():
    print(f"  {val!r:25s} → {cnt}")

fig = go.Figure(go.Bar(
    x=gender_vc.index.tolist(), y=gender_vc.values,
    marker_color=CATEGORICAL_COLORS[:len(gender_vc)],
    text=gender_vc.values, textposition="outside",
))
styled(fig, "Gender — All Unique Encodings (Before Cleaning)")
fig.update_xaxes(title_text="Gender Value")
fig.update_yaxes(title_text="Count")
fig.show()

Gender value counts:
  'Female'                  → 206
  'Male'                    → 201
  'Non-binary'              → 34
  'Prefer not to say'       → 19
  'M'                       → 12
  'F'                       → 12
  'man'                     → 11
  'FEMALE'                  → 10
  'male'                    → 10
  'female'                  → 7
  'MALE'                    → 7
  'woman'                   → 6


**Canonical mapping:**

| Variant | Maps to |
|---------|--------|
| M, male, MALE, man | Male |
| F, female, FEMALE, woman | Female |

**So what?** Gender is nearly perfectly balanced after cleaning (Male ≈ 241, Female ≈ 241, Non-binary = 34, Prefer not to say = 19). Leaving uncleaned would fragment "Male" into 5 categories, destroying grouped analysis.

### 5.3 gpa — Out-of-range values

8 values outside [0, 4.0] plus 42 missing (7.85%).

In [13]:
gpa_vals = pd.to_numeric(df["gpa"], errors="coerce")
bad_gpa = gpa_vals[(gpa_vals < 0) | (gpa_vals > 4)]

print(f"Out-of-range GPA values: {sorted(bad_gpa.dropna().unique().tolist())}")
print(f"Rows affected: {len(bad_gpa.dropna())}")
print(f"Missing GPA: {gpa_vals.isna().sum()} ({gpa_vals.isna().mean()*100:.1f}%)")

fig = go.Figure(go.Histogram(x=gpa_vals, nbinsx=40, marker_color=PALETTE["accent1"]))
for v in bad_gpa.dropna().unique():
    fig.add_vline(x=v, line_dash="dash", line_color="#F4A8B0",
                  annotation_text=f"out-of-range: {v}")
styled(fig, "GPA Distribution (flagging out-of-range)")
fig.update_xaxes(title_text="GPA")
fig.update_yaxes(title_text="Count")
fig.show()

Out-of-range GPA values: [-1.0, -0.5, 4.75, 4.8, 4.99, 5.2, 5.5, 6.0]
Rows affected: 8
Missing GPA: 42 (7.9%)


**So what?** GPA is a key outcome variable. 7.85% missing + 8 impossible = ~9.3% of rows need attention before any GPA-based analysis.

### 5.4 study_hours_per_day — Impossible values

5 impossible values (negative or > 16 hours/day).

In [14]:
sh = df["study_hours_per_day"]
sh_bad = sh[(sh < 0) | (sh > 16)]
print(f"Impossible study hours: {sorted(sh_bad.dropna().unique().tolist())}")
print(f"Rows affected: {len(sh_bad)}")

fig = go.Figure(go.Histogram(x=sh, nbinsx=40, marker_color=PALETTE["primary"]))
for v in sh_bad.dropna().unique():
    fig.add_vline(x=v, line_dash="dash", line_color="#F4A8B0",
                  annotation_text=f"impossible: {v}")
styled(fig, "Study Hours per Day Distribution")
fig.update_xaxes(title_text="Hours / Day")
fig.update_yaxes(title_text="Count")
fig.show()

Impossible study hours: [-2.0, 25.0, 28.0, 30.0, 35.0]
Rows affected: 5


### 5.5 attendance_rate — Values exceeding 100%

10 values exceed 100%, with maximum at 128%.

In [15]:
ar = df["attendance_rate"]
ar_bad = ar[ar > 100]
print(f"Attendance > 100%: {len(ar_bad)} rows")
print(f"Values: {sorted(ar_bad.unique().tolist())}")

fig = go.Figure(go.Histogram(x=ar, nbinsx=40, marker_color=PALETTE["accent2"]))
for v in ar_bad.dropna().unique():
    fig.add_vline(x=v, line_dash="dash", line_color="#F4A8B0",
                  annotation_text=f">100%: {v}")
styled(fig, "Attendance Rate Distribution")
fig.update_xaxes(title_text="Attendance (%)")
fig.update_yaxes(title_text="Count")
fig.show()

Attendance > 100%: 10 rows
Values: [103.0, 112.0, 116.0, 118.0, 121.0, 122.0, 124.0, 126.0, 127.0, 128.0]


**So what?** Percentages > 100 are impossible. These likely represent data entry errors. Setting to NaN is the safest approach.

### 5.6 sleep_hours_per_night — Extreme values

4 extreme values (negative or > 16 hrs) plus 53 missing (9.91%).

In [16]:
sleep_vals = pd.to_numeric(df["sleep_hours_per_night"], errors="coerce")
bad_sleep = sleep_vals[(sleep_vals < 0) | (sleep_vals > 16)]
print(f"Extreme sleep values: {sorted(bad_sleep.dropna().unique().tolist())}")
print(f"Rows affected: {len(bad_sleep.dropna())}")
print(f"Missing: {sleep_vals.isna().sum()} ({sleep_vals.isna().mean()*100:.1f}%)")

fig = go.Figure(go.Histogram(x=sleep_vals, nbinsx=40, marker_color=PALETTE["accent3"]))
for v in bad_sleep.dropna().unique():
    fig.add_vline(x=v, line_dash="dash", line_color="#F4A8B0",
                  annotation_text=f"extreme: {v}")
styled(fig, "Sleep Hours Distribution")
fig.update_xaxes(title_text="Hours per Night")
fig.update_yaxes(title_text="Count")
fig.show()

Extreme sleep values: [-3.0, -1.0, 22.0, 24.0]
Rows affected: 4
Missing: 53 (9.9%)


### 5.7 stress_level — Mixed types (text + numeric)

20 rows contain text labels ("low", "medium", "high", "very high") instead of numeric values.

In [17]:
stress_numeric = pd.to_numeric(df["stress_level"], errors="coerce")
stress_text = df["stress_level"][stress_numeric.isna() & df["stress_level"].notna()]
print(f"Text entries in stress_level: {len(stress_text)}")
print(f"Text value counts:\n{stress_text.value_counts().to_string()}")

stress_all = df["stress_level"].dropna()
stress_vc = stress_all.value_counts().head(20)
fig = go.Figure(go.Bar(
    x=[str(v) for v in stress_vc.index], y=stress_vc.values,
    marker_color=[PALETTE["secondary"] if not str(v).replace('.','').replace('-','').isdigit()
                  else PALETTE["primary"] for v in stress_vc.index],
    text=stress_vc.values, textposition="outside",
))
styled(fig, "Stress Level Values (text entries highlighted in rose)")
fig.update_xaxes(title_text="Stress Level Value")
fig.update_yaxes(title_text="Count")
fig.show()

Text entries in stress_level: 20
Text value counts:
stress_level
low          8
medium       6
high         4
very high    2


**So what?** This column cannot be used in any numeric analysis until the text labels are mapped to numbers.

**Suggested mapping:** low → 2, medium → 5, high → 8, very high → 9.5

### 5.8 on_campus — Inconsistent encoding

60 rows (11.2%) use non-canonical encodings across 10 variants.

In [18]:
oc_vc = df["on_campus"].value_counts()
print("on_campus value counts:")
for val, cnt in oc_vc.items():
    print(f"  {val!r:15s} → {cnt}")

fig = go.Figure(go.Bar(
    x=[str(v) for v in oc_vc.index], y=oc_vc.values,
    marker_color=CATEGORICAL_COLORS[:len(oc_vc)],
    text=oc_vc.values, textposition="outside",
))
styled(fig, "on_campus — All Unique Encodings")
fig.update_xaxes(title_text="on_campus value")
fig.update_yaxes(title_text="Count")
fig.show()

on_campus value counts:
  'True'          → 258
  'False'         → 217
  'YES'           → 13
  'true'          → 8
  '1'             → 7
  '0'             → 7
  'false'         → 7
  'yes'           → 7
  'NO'            → 6
  'no'            → 5


**So what?** After mapping all variants to boolean True/False: True ≈ 293, False ≈ 242.

### 5.9 caffeine_mg_per_day

Range is plausible (0–472 mg). 37 missing values (6.9%).

In [19]:
caf_vals = pd.to_numeric(df["caffeine_mg_per_day"], errors="coerce")
print(f"Range: {caf_vals.min():.0f} – {caf_vals.max():.0f} mg")
print(f"Missing: {caf_vals.isna().sum()} ({caf_vals.isna().mean()*100:.1f}%)")

fig = go.Figure(go.Histogram(x=caf_vals, nbinsx=40, marker_color=PALETTE["accent2"]))
styled(fig, "Caffeine Intake Distribution (mg/day)")
fig.update_xaxes(title_text="Caffeine (mg)")
fig.update_yaxes(title_text="Count")
fig.show()

Range: 0 – 472 mg
Missing: 37 (6.9%)


### 5.10 monthly_spending

Range $200–$1709. 48 missing values (9%).

In [20]:
ms_vals = pd.to_numeric(df["monthly_spending"], errors="coerce")
print(f"Range: ${ms_vals.min():.0f} – ${ms_vals.max():.0f}")
print(f"Mean: ${ms_vals.mean():.2f}  |  Median: ${ms_vals.median():.2f}")
print(f"Missing: {ms_vals.isna().sum()} ({ms_vals.isna().mean()*100:.1f}%)")

fig = go.Figure(go.Histogram(x=ms_vals, nbinsx=40, marker_color=PALETTE["accent1"]))
styled(fig, "Monthly Spending Distribution ($)")
fig.update_xaxes(title_text="Spending ($)")
fig.update_yaxes(title_text="Count")
fig.show()

Range: $200 – $1709
Mean: $834.65  |  Median: $817.62
Missing: 48 (9.0%)


## 6. Full Data Quality Issue Inventory

All 16 issues flagged by the diagnostic, with recommendations.

In [21]:
issues = []

def flag(col, issue_type, affected, recommendation):
    issues.append({"column": col, "issue_type": issue_type,
                   "affected_rows": affected, "recommendation": recommendation})

# Duplicates
flag("(whole row)", "duplicate_rows", n_dups,
     "Drop exact duplicate rows, keep first occurrence")

# Age
flag("age", "impossible_value", int(((pd.to_numeric(df['age'], errors='coerce') < 16) |
     (pd.to_numeric(df['age'], errors='coerce') > 60)).sum()),
     "Set impossible ages (< 16 or > 60) to NaN; impute with median later")

# Gender
canonical_gender = {"Male", "Female", "Non-binary", "Prefer not to say"}
flag("gender", "inconsistent_encoding",
     int((~df['gender'].isin(canonical_gender)).sum()),
     "Map all variants to canonical: Male/Female/Non-binary/Prefer not to say")

# GPA
gv = pd.to_numeric(df['gpa'], errors='coerce')
flag("gpa", "impossible_value", int(((gv < 0) | (gv > 4)).sum()),
     "Set values outside [0, 4] to NaN")

# Study hours
flag("study_hours_per_day", "impossible_value",
     int(((df['study_hours_per_day'] < 0) | (df['study_hours_per_day'] > 16)).sum()),
     "Set values outside [0, 16] to NaN")

# Attendance
flag("attendance_rate", "impossible_value",
     int((df['attendance_rate'] > 100).sum()),
     "Set > 100 to NaN")

# Sleep
sv = pd.to_numeric(df['sleep_hours_per_night'], errors='coerce')
flag("sleep_hours_per_night", "impossible_value",
     int(((sv < 0) | (sv > 16)).sum()),
     "Set extreme values to NaN")

# Stress
sn = pd.to_numeric(df['stress_level'], errors='coerce')
flag("stress_level", "mixed_types",
     int((sn.isna() & df['stress_level'].notna()).sum()),
     "Map text→numeric (low→2, medium→5, high→8, very high→9.5)")

# on_campus
flag("on_campus", "inconsistent_encoding",
     int((~df['on_campus'].isin({'True', 'False'})).sum()),
     "Map all variants to boolean True/False")

# Missing values
for col in df.columns:
    n_miss = int(df[col].isna().sum())
    if n_miss > 0 and col not in [i['column'] for i in issues]:
        flag(col, "missing_values", n_miss,
             "Investigate missingness pattern; decide imputation in Phase 1")

issues_df = pd.DataFrame(issues)
issues_df

,column,issue_type,affected_rows,recommendation
0,(whole row),duplicate_rows,3,"Drop exact duplicate rows, keep first occurrence"
1,age,impossible_value,6,Set impossible ages (< 16 or > 60) to NaN; imp...
2,gender,inconsistent_encoding,75,Map all variants to canonical: Male/Female/Non...
3,gpa,impossible_value,8,"Set values outside [0, 4] to NaN"
4,study_hours_per_day,impossible_value,5,"Set values outside [0, 16] to NaN"
5,attendance_rate,impossible_value,10,Set > 100 to NaN
6,sleep_hours_per_night,impossible_value,4,Set extreme values to NaN
7,stress_level,mixed_types,20,"Map text→numeric (low→2, medium→5, high→8, ver..."
8,on_campus,inconsistent_encoding,60,Map all variants to boolean True/False
9,exercise_days_per_week,missing_values,32,Investigate missingness pattern; decide imputa...


## 7. Cleaning Decision Log

| # | Column | Issue | Decision | Rationale |
|---|--------|-------|----------|-----------|
| 1 | (whole row) | 3 exact duplicate rows | Drop duplicates, keep first | Duplicates inflate sample and bias stats |
| 2 | age | 6 impossible values (−3, 0, 5, 150, 200, 999) | Set to NaN; impute with median (21) | Clearly data entry errors |
| 3 | gender | 75 rows with variant encodings | Map to canonical 4 values | Need consistent categories |
| 4 | gpa | 8 values outside [0, 4.0] | Set to NaN | Impossible on a 4.0 scale |
| 5 | study_hours_per_day | 5 impossible values | Set to NaN | Cannot study negative or > 16 hrs |
| 6 | attendance_rate | 10 values > 100% | Set to NaN | Percentages can't exceed 100 |
| 7 | sleep_hours_per_night | 4 extreme values | Set to NaN | Cannot sleep negative or 24 hrs |
| 8 | stress_level | 20 text labels | Map: low→2, medium→5, high→8, very high→9.5 | Need numeric scale |
| 9 | on_campus | 60 rows non-canonical | Map to boolean True/False | Need consistent binary |
| 10–18 | 9 columns | Missing values (397 total) | Keep as NaN; evaluate imputation in Phase 1 | Different columns need different strategies |

## 8. Key Questions for Phase 1

1. **Missingness pattern:** Do anxiety_score and depression_score always go missing together? Is missingness correlated with stress, major, or year?
2. **Stress text labels:** Is the mapping low→2, medium→5, high→8, very high→9.5 the best choice? Sensitivity analysis may be needed.
3. **Attendance > 100%:** Could these represent extra-credit adjustments, or are they pure errors?
4. **Age distribution:** After removing impossibles, is the 18–25 range truly complete or are some ages dominant?
5. **GPA missingness:** Is GPA more likely to be missing for certain majors or year levels?